In [1]:
import pandas as pd

providers = pd.read_csv("medical.csv")

providers.shape

C:\Users\baoxi\AppData\Local\Temp\ipykernel_9552\510198739.py:3: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  providers = pd.read_csv("medical.csv")


(374831, 46)

In [2]:
providers.head()

,_id,Provider_Number,NPI,Owner_Number,Service_Location_Number,Legal_Name,Enroll_Status_Eff_DT,Provider_Taxonomy,NEMT_NMT_Provider_Type,ANC_Provider_Type,...,Acc_Parking,Acc_Restroom,Acc_Medical_Equipment,Acc_Patient_Areas,Acc_Patient_Diagnostic,Provider_Source,Billing_Provider,Latitude,Longitude,CountyName
0,1,1003000225,1.003000e+09,1,1,"MORGAN, ANNA C A MD",10/15/2008,No Taxonomy Provided,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,1,FFS Non-Billing Provider,37.891776,-122.058050,Contra Costa County
1,2,1003000563,1.003001e+09,1,1,"CHUN, RICHARD B D MD INC",4/1/2002,207RG0100X,NaN,Gastroenterology,...,NaN,NaN,NaN,NaN,NaN,1,FFS Billing Provider,38.734427,-121.271995,Placer County
2,3,1003000563,1.003001e+09,1,1,"CHUN, RICHARD B D MD INC",4/1/2002,207RG0100X,NaN,Gastroenterology,...,NaN,NaN,NaN,NaN,NaN,1,FFS Billing Provider,38.734427,-121.271995,Placer County
3,4,1003000639,1.003001e+09,1,1,"BENHARASH, PEYMAN MD",1/19/2011,No Taxonomy Provided,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,1,FFS Non-Billing Provider,34.063722,-118.443467,Los Angeles County
4,5,1003001124,1.003001e+09,0,0,"HOWELL, MYLA LCSW",7/15/2024,Taxonomy not applicable,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,3,FFS Non-Billing Provider,34.064899,-118.291625,Los Angeles County


In [3]:
print("Rows:", len(providers))

print(
    "Unique providers:",
    providers["Provider_Number"].nunique()
)

print(
    "Unique provider-location combinations:",
    providers[
        ["Provider_Number", "Service_Location_Number"]
    ].drop_duplicates().shape[0]
)

print(
    "Unique _id:",
    providers["_id"].nunique()
)

Rows: 374831
Unique providers: 305145
Unique provider-location combinations: 330006
Unique _id: 374831


In [4]:
providers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 374831 entries, 0 to 374830
Data columns (total 46 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   _id                       374831 non-null  int64  
 1   Provider_Number           374831 non-null  object 
 2   NPI                       374782 non-null  float64
 3   Owner_Number              374831 non-null  int64  
 4   Service_Location_Number   374831 non-null  int64  
 5   Legal_Name                374831 non-null  object 
 6   Enroll_Status_Eff_DT      374831 non-null  object 
 7   Provider_Taxonomy         374831 non-null  object 
 8   NEMT_NMT_Provider_Type    1364 non-null    object 
 9   ANC_Provider_Type         163072 non-null  object 
 10  FI_Provider_Type_CD       348847 non-null  float64
 11  FI_Provider_Type          348847 non-null  object 
 12  Provider_License          339395 non-null  object 
 13  FI_Provider_Specialty_CD  225865 non-null  o

In [5]:
key_columns = [
    "Provider_Number",
    "Service_Location_Number",
    "CountyName",
    "FI_Provider_Specialty",
    "Telehealth_Services",
    "Threshold_Languages",
    "Latitude",
    "Longitude"
]

providers[key_columns].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 374831 entries, 0 to 374830
Data columns (total 8 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   Provider_Number          374831 non-null  object 
 1   Service_Location_Number  374831 non-null  int64  
 2   CountyName               372355 non-null  object 
 3   FI_Provider_Specialty    225865 non-null  object 
 4   Telehealth_Services      2983 non-null    object 
 5   Threshold_Languages      2983 non-null    object 
 6   Latitude                 374831 non-null  float64
 7   Longitude                374831 non-null  float64
dtypes: float64(2), int64(1), object(5)
memory usage: 22.9+ MB


In [6]:
providers[key_columns].isna().mean() * 100

Provider_Number             0.000000
Service_Location_Number     0.000000
CountyName                  0.660564
FI_Provider_Specialty      39.742177
Telehealth_Services        99.204175
Threshold_Languages        99.204175
Latitude                    0.000000
Longitude                   0.000000
dtype: float64

In [7]:
providers[providers["CountyName"].isna()][
    ["Provider_Number", "Service_Location_Number", "CountyName", "Latitude", "Longitude"]
].head(10)

,Provider_Number,Service_Location_Number,CountyName,Latitude,Longitude
590,1003156944,1,NaN,0.0,0.0
2768,1003397852,0,NaN,0.0,0.0
3279,1003506221,1,NaN,0.0,0.0
3280,1003506221,2,NaN,0.0,0.0
3531,1003604513,1,NaN,0.0,0.0
3543,1003617945,1,NaN,0.0,0.0
3551,1003620311,1,NaN,0.0,0.0
3570,1003635707,1,NaN,0.0,0.0
3572,1003636499,1,NaN,0.0,0.0
3585,1003642463,1,NaN,0.0,0.0


In [8]:
missing_county = providers[providers["CountyName"].isna()]

print("Missing county rows:", len(missing_county))
print("Missing latitude:", missing_county["Latitude"].isna().sum())
print("Missing longitude:", missing_county["Longitude"].isna().sum())

Missing county rows: 2476
Missing latitude: 0
Missing longitude: 0


In [9]:
clean = providers.copy()

In [10]:
clean["FI_Provider_Specialty"] = clean["FI_Provider_Specialty"].fillna("Unknown")

print("Rows:", len(clean))
print("Missing specialty:", clean["FI_Provider_Specialty"].isna().sum())

Rows: 374831
Missing specialty: 0


In [11]:
duplicate_check = clean.duplicated(
    subset=["Provider_Number", "Service_Location_Number"],
    keep=False
)

print("Rows in repeated provider-location combinations:", duplicate_check.sum())

clean[duplicate_check][
    ["Provider_Number", "Service_Location_Number", "FI_Provider_Specialty", "CountyName"]
].head(20)

Rows in repeated provider-location combinations: 71273


,Provider_Number,Service_Location_Number,FI_Provider_Specialty,CountyName
1,1003000563,1,Gastroenterology (M.D. only),Placer County
2,1003000563,1,Internal Medicine,Placer County
22,1003008855,0,Unknown,Tulare County
23,1003008855,0,Unknown,Kern County
37,1003012733,1,Pediatrics,San Diego County
38,1003012733,1,Internal Medicine,San Diego County
41,1003013269,0,Unknown,Santa Cruz County
42,1003013269,0,Unknown,Santa Cruz County
44,1003013319,1,Pediatrics,Adams County
45,1003013319,1,Emergency Medicine (Urgent Care),Adams County


In [12]:
clean_locations = clean.drop_duplicates(
    subset=["Provider_Number", "Service_Location_Number"]
).copy()

print("Rows before:", len(clean))
print("Rows after:", len(clean_locations))
print(
    "Duplicate provider-location rows left:",
    clean_locations.duplicated(
        subset=["Provider_Number", "Service_Location_Number"]
    ).sum()
)

Rows before: 374831
Rows after: 330006
Duplicate provider-location rows left: 0


In [13]:
print(clean_locations["CountyName"].value_counts().head(10))

print()

print(clean_locations["FI_Provider_Specialty"].value_counts().head(10))

CountyName
Los Angeles County       83332
San Diego County         24724
Orange County            24447
Santa Clara County       23262
San Bernardino County    16610
Sacramento County        15831
San Francisco County     15827
Riverside County         15407
Alameda County           13519
Contra Costa County       7900
Name: count, dtype: int64

FI_Provider_Specialty
Unknown                                       142867
NOT APPLICABLE                                 73618
Internal Medicine                              20576
Pediatrics                                     10129
Family Practice                                10103
Emergency Medicine (Urgent Care)                8535
General Practice                                7954
Anesthesiology                                  7096
Radiology                                       6604
Obstetrics-Gynecology (M.D. Only) Neonatal      4867
Name: count, dtype: int64


In [14]:
clean_locations = clean_locations[
    ~clean_locations["FI_Provider_Specialty"].isin(["Unknown", "NOT APPLICABLE"])
].copy()

clean_locations["FI_Provider_Specialty"].value_counts().head(10)


FI_Provider_Specialty
Internal Medicine                             20576
Pediatrics                                    10129
Family Practice                               10103
Emergency Medicine (Urgent Care)               8535
General Practice                               7954
Anesthesiology                                 7096
Radiology                                      6604
Obstetrics-Gynecology (M.D. Only) Neonatal     4867
General Surgery                                4083
Psychiatry                                     3571
Name: count, dtype: int64

In [15]:
clean_locations["CountyName"] = clean_locations["CountyName"].fillna("Unknown")

clean_locations["CountyName"].isna().sum()

np.int64(0)

In [16]:
clean_locations.shape

clean_locations[["Provider_Number", "Service_Location_Number", "CountyName",
                 "FI_Provider_Specialty", "Latitude", "Longitude"]].head()

,Provider_Number,Service_Location_Number,CountyName,FI_Provider_Specialty,Latitude,Longitude
0,1003000225,1,Contra Costa County,Obstetrics-Gynecology (M.D. Only) Neonatal,37.891776,-122.058050
1,1003000563,1,Placer County,Gastroenterology (M.D. only),38.734427,-121.271995
3,1003000639,1,Los Angeles County,Thoracic Surgery,34.063722,-118.443467
5,1003001157,1,San Bernardino County,Pediatrics,34.099394,-117.421262
7,1003001363,1,Imperial County,Anesthesiology,32.778685,-115.569497


In [17]:
clean_locations.to_csv("cleaned_provider_locations.csv", index=False)

In [18]:
clean_locations.shape

clean_locations.isna().sum()

_id                              0
Provider_Number                  0
NPI                              0
Owner_Number                     0
Service_Location_Number          0
Legal_Name                       0
Enroll_Status_Eff_DT             0
Provider_Taxonomy                0
NEMT_NMT_Provider_Type      113521
ANC_Provider_Type            64164
FI_Provider_Type_CD              0
FI_Provider_Type                 0
Provider_License              7562
FI_Provider_Specialty_CD         0
FI_Provider_Specialty            0
Out_of_State_Indicator           0
In_Out_State                     0
Address_Attention           105343
Address                          0
Address2                     78337
City                             0
State                            0
ZIP                              0
ZIP_4                            0
DHCS_County_CD                   0
FIPS_County_CD                3063
Phone_Number                 95662
Medicaid_Patients           113467
CHIP_Patients       

In [19]:
clean_locations[[
    "Provider_Number",
    "Service_Location_Number",
    "CountyName",
    "FI_Provider_Specialty",
    "Telehealth_Services",
    "Threshold_Languages",
    "Latitude",
    "Longitude"
]].isna().sum()

Provider_Number                 0
Service_Location_Number         0
CountyName                      0
FI_Provider_Specialty           0
Telehealth_Services        113153
Threshold_Languages        113153
Latitude                        0
Longitude                       0
dtype: int64

In [20]:
clean_locations["Telehealth_Services"] = clean_locations["Telehealth_Services"].fillna("Unknown")
clean_locations["Threshold_Languages"] = clean_locations["Threshold_Languages"].fillna("Unknown")

clean_locations[[
    "Provider_Number",
    "Service_Location_Number",
    "CountyName",
    "FI_Provider_Specialty",
    "Telehealth_Services",
    "Threshold_Languages",
    "Latitude",
    "Longitude"
]].isna().sum()

Provider_Number            0
Service_Location_Number    0
CountyName                 0
FI_Provider_Specialty      0
Telehealth_Services        0
Threshold_Languages        0
Latitude                   0
Longitude                  0
dtype: int64

In [21]:
clean_locations.to_csv("cleaned_provider_locations.csv", index=False)

In [22]:
clean_locations.shape
clean_locations.head()

,_id,Provider_Number,NPI,Owner_Number,Service_Location_Number,Legal_Name,Enroll_Status_Eff_DT,Provider_Taxonomy,NEMT_NMT_Provider_Type,ANC_Provider_Type,...,Acc_Parking,Acc_Restroom,Acc_Medical_Equipment,Acc_Patient_Areas,Acc_Patient_Diagnostic,Provider_Source,Billing_Provider,Latitude,Longitude,CountyName
0,1,1003000225,1.003000e+09,1,1,"MORGAN, ANNA C A MD",10/15/2008,No Taxonomy Provided,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,1,FFS Non-Billing Provider,37.891776,-122.058050,Contra Costa County
1,2,1003000563,1.003001e+09,1,1,"CHUN, RICHARD B D MD INC",4/1/2002,207RG0100X,NaN,Gastroenterology,...,NaN,NaN,NaN,NaN,NaN,1,FFS Billing Provider,38.734427,-121.271995,Placer County
3,4,1003000639,1.003001e+09,1,1,"BENHARASH, PEYMAN MD",1/19/2011,No Taxonomy Provided,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,1,FFS Non-Billing Provider,34.063722,-118.443467,Los Angeles County
5,6,1003001157,1.003001e+09,1,1,"KIM, DAVID C MD",7/18/2003,No Taxonomy Provided,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,1,FFS Non-Billing Provider,34.099394,-117.421262,San Bernardino County
7,8,1003001363,1.003001e+09,2,1,"STEVENS, CHARLES R MD",8/16/2009,No Taxonomy Provided,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,1,FFS Non-Billing Provider,32.778685,-115.569497,Imperial County


In [23]:
clean_locations.shape

(113521, 46)

In [24]:
clean_locations.to_csv("cleaned_provider_locations.csv", index=False)